In [1]:
import json
import random
import sys
from datetime import datetime

try:
    import tkinter as tk
    from tkinter import scrolledtext
except ImportError:
    tk = None  # Online environment may not have GUI

try:
    import speech_recognition as sr
    import pyttsx3
except ImportError:
    sr = None
    pyttsx3 = None

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB

# ---------------- Load intents ----------------
with open('intents.json', 'r') as f:
    data = json.load(f)

X, y = [], []
for intent in data['intents']:
    for pattern in intent['patterns']:
        X.append(pattern)
        y.append(intent['tag'])

vectorizer = CountVectorizer()
X_vec = vectorizer.fit_transform(X)
model = MultinomialNB()
model.fit(X_vec, y)

# ---------------- Text-to-speech ----------------
if pyttsx3:
    engine = pyttsx3.init()
    engine.setProperty('rate', 160)
    engine.setProperty('voice', engine.getProperty('voices')[0].id)

    def speak(text):
        engine.say(text)
        engine.runAndWait()
else:
    def speak(text):
        pass  # skip if pyttsx3 not available

# ---------------- Chatbot response ----------------
def get_response(user_input):
    user_vec = vectorizer.transform([user_input])
    tag = model.predict(user_vec)[0]
    for intent in data['intents']:
        if intent['tag'] == tag:
            # Replace {{time}} if present
            responses = intent['responses']
            reply = random.choice(responses)
            if "{{time}}" in reply:
                reply = reply.replace("{{time}}", datetime.now().strftime("%H:%M"))
            return reply

# ---------------- Offline GUI mode ----------------
def run_gui():
    root = tk.Tk()
    root.title("💬 Voice-Enabled Chatbot")
    root.geometry("520x600")
    root.configure(bg="#1E1E1E")

    frame_chat = tk.Frame(root, bg="#1E1E1E")
    frame_chat.pack(pady=10, padx=10, fill="both", expand=True)

    canvas = tk.Canvas(frame_chat, bg="#1E1E1E", highlightthickness=0)
    scrollbar = tk.Scrollbar(frame_chat, command=canvas.yview)
    scrollable_frame = tk.Frame(canvas, bg="#1E1E1E")

    scrollable_frame.bind("<Configure>", lambda e: canvas.configure(scrollregion=canvas.bbox("all")))
    canvas.create_window((0, 0), window=scrollable_frame, anchor="nw")
    canvas.configure(yscrollcommand=scrollbar.set)
    canvas.pack(side="left", fill="both", expand=True)
    scrollbar.pack(side="right", fill="y")

    entry = tk.Entry(root, font=("Segoe UI", 13), bg="#2D2D2D", fg="white", insertbackground="white", relief="flat")
    entry.pack(side="left", padx=(15, 5), pady=10, fill="x", expand=True)

    def create_bubble(message, sender="user"):
        time_str = datetime.now().strftime("%H:%M")
        frame = tk.Frame(scrollable_frame, bg="#1E1E1E")
        frame.pack(anchor="e" if sender=="user" else "w", pady=5, padx=10)
        color = "#0078D7" if sender == "user" else "#333333"
        text_color = "white" if sender == "user" else "#E0E0E0"
        label = tk.Label(frame, text=message, bg=color, fg=text_color, wraplength=350, justify="left",
                         font=("Segoe UI", 11), padx=10, pady=6)
        label.pack(anchor="e" if sender=="user" else "w")
        tk.Label(frame, text=time_str, font=("Segoe UI", 8), fg="#888888", bg="#1E1E1E").pack(
            anchor="e" if sender=="user" else "w", padx=5)
        root.after(100, lambda: canvas.yview_moveto(1.0))

    def process_message(user_text):
        create_bubble(user_text, sender="user")
        bot_reply = get_response(user_text)
        root.after(400, lambda: (create_bubble(bot_reply, sender="bot"), speak(bot_reply)))

    def send_message(event=None):
        user_text = entry.get().strip()
        if not user_text:
            return
        entry.delete(0, tk.END)
        process_message(user_text)

    def listen_voice():
        if sr is None:
            create_bubble("Speech recognition not available.", sender="bot")
            return
        recognizer = sr.Recognizer()
        with sr.Microphone() as source:
            create_bubble("🎤 Listening...", sender="bot")
            audio = recognizer.listen(source, timeout=5, phrase_time_limit=5)
        try:
            text = recognizer.recognize_google(audio)
            create_bubble(f"(You said): {text}", sender="user")
            process_message(text)
        except sr.UnknownValueError:
            create_bubble("❌ Sorry, I couldn't understand.", sender="bot")
        except sr.RequestError:
            create_bubble("⚠️ Speech recognition not available offline.", sender="bot")

    send_btn = tk.Button(root, text="Send", font=("Segoe UI", 11, "bold"),
                         bg="#0078D7", fg="white", activebackground="#005A9E",
                         relief="flat", command=send_message)
    send_btn.pack(side="right", padx=(5, 10), pady=10)

    mic_btn = tk.Button(root, text="🎤 Speak", font=("Segoe UI", 11, "bold"),
                        bg="#28A745", fg="white", activebackground="#1F7A32",
                        relief="flat", command=listen_voice)
    mic_btn.pack(side="right", padx=(5, 5), pady=10)

    root.bind('<Return>', send_message)
    create_bubble("Hello! You can type or speak to me 😊", sender="bot")
    root.mainloop()

# ---------------- Online / CLI mode ----------------
def run_cli():
    print("🤖 Chatbot is ready! (type 'exit' to quit)")
    while True:
        try:
            user = input("You: ")
        except (EOFError, KeyboardInterrupt):
            print("\nExiting...")
            break
        if user.lower() in ["exit", "quit"]:
            print("Chatbot: Goodbye!")
            break
        bot_reply = get_response(user)
        print("Chatbot:", bot_reply)
        speak(bot_reply)

# ---------------- Main ----------------
if __name__ == "__main__":
    if tk is not None:
        run_gui()
    else:
        run_cli()


Exception in Tkinter callback
Traceback (most recent call last):
  File "c:\Users\nazmu\AppData\Local\Programs\Python\Python313\Lib\site-packages\speech_recognition\__init__.py", line 103, in get_pyaudio
    import pyaudio
ModuleNotFoundError: No module named 'pyaudio'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "c:\Users\nazmu\AppData\Local\Programs\Python\Python313\Lib\tkinter\__init__.py", line 2074, in __call__
    return self.func(*args)
           ~~~~~~~~~^^^^^^^
  File "C:\Users\nazmu\AppData\Local\Temp\ipykernel_6988\1602525782.py", line 116, in listen_voice
    with sr.Microphone() as source:
         ~~~~~~~~~~~~~^^
  File "c:\Users\nazmu\AppData\Local\Programs\Python\Python313\Lib\site-packages\speech_recognition\__init__.py", line 75, in __init__
    self.pyaudio_module = self.get_pyaudio()
                          ~~~~~~~~~~~~~~~~^^
  File "c:\Users\nazmu\AppData\Local\Programs\Python\Python313\Lib\site-